# Prompt Evaluation for AI Agents
This notebook demonstrates how to evaluate different prompts for a physics teacher agent.

In [1]:
# Setup - Install dependencies
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 3.4 MB/s eta 0:00:00


In [2]:
import getpass
import os
from groq import Groq

# Get API key securely
os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ API key: ")
client = Groq()

Enter your GROQ API key: ··········


## 1. Define Evaluation Metrics (Binary)
We'll use 3 binary metrics (Pass/Fail) to evaluate our physics teacher agent:
- **Accuracy**: Is the answer factually correct with no misconceptions?
- **Clarity**: Is the explanation clear enough for a confused student?
- **Completeness**: Does it address the nuance/trap in the question?

In [3]:
import re

def evaluate_response(question, expected_answer, actual_response):
    """Use LLM to evaluate response with binary scores (0 or 1)"""
    eval_prompt = f"""Evaluate this physics teacher response with STRICT binary scoring (0=Fail, 1=Pass).

Question: {question}
Expected Key Points: {expected_answer}
Actual Response: {actual_response}

Be STRICT. Only give 1 if the criterion is fully met:
- Accuracy: Is it factually correct with NO errors or misleading statements? (0 or 1)
- Clarity: Would a confused student understand this WITHOUT further questions? (0 or 1)
- Completeness: Does it address the SPECIFIC nuance/trap in the question? (0 or 1)

Respond ONLY in this exact format:
Accuracy: X
Clarity: X
Completeness: X"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": eval_prompt}],
        temperature=0
    )

    text = response.choices[0].message.content
    scores = {}
    for metric in ["Accuracy", "Clarity", "Completeness"]:
        match = re.search(rf"{metric}\s*:\s*([01])", text, re.IGNORECASE)
        scores[metric] = int(match.group(1)) if match else 0
    return scores

## 2. Physics Teacher Agent

In [4]:
def physics_teacher(system_prompt, question):
    """Simple agent that answers physics questions based on the given prompt"""
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        temperature=0.3
    )
    return response.choices[0].message.content

## 3. Synthetic Dataset (10 Challenging Q&A)
These questions target common misconceptions and edge cases that require careful explanation.

In [5]:
dataset = [
    {"q": "If I drop a bowling ball and a feather from the same height, which hits the ground first and why?",
     "a": "Must clarify: In air, bowling ball wins due to air resistance. In a vacuum, they hit simultaneously. Must explain the difference and why mass doesn't affect fall rate in vacuum (F=ma, heavier objects have proportionally more force)."},

    {"q": "Why don't satellites fall down to Earth?",
     "a": "They ARE constantly falling! They just move sideways fast enough that they keep missing Earth. Must explain orbital mechanics - falling toward Earth while moving tangentially. Avoid saying 'no gravity in space'."},

    {"q": "If you're in a car moving at 60 mph and throw a ball forward at 10 mph, how fast is the ball going?",
     "a": "Must specify: 70 mph relative to ground, 10 mph relative to car. Must explain reference frames clearly. This is Galilean relativity."},

    {"q": "Does a heavy truck push harder on a car during a collision than the car pushes on the truck?",
     "a": "No! Newton's 3rd law - forces are equal and opposite. The truck causes more DAMAGE because the car accelerates more (F=ma, smaller mass = larger acceleration). Must distinguish force from effect."},
    {"q": "Why do astronauts float in the space station? Is there no gravity there?",
     "a": "Gravity at ISS altitude is ~90% of surface gravity! Astronauts float because they're in continuous free-fall (orbit). Must correct the 'no gravity' misconception."},

    {"q": "If you spin a bucket of water over your head, why doesn't the water fall out at the top?",
     "a": "Must explain: water's inertia makes it want to go straight, bucket pushes it in a circle. At top, gravity + bucket provide centripetal force. If spun too slowly, water WILL fall. Include the speed threshold concept."},

    {"q": "Does hot water freeze faster than cold water?",
     "a": "Sometimes yes (Mpemba effect), but it's complex and not fully understood. Depends on container, environment, dissolved gases. Must avoid oversimplifying - acknowledge the nuance and ongoing research."},

    {"q": "Why can't you go faster than light? What happens if you try?",
     "a": "As you approach c, your relativistic mass increases, requiring infinite energy to reach c. Time dilation and length contraction occur. Must explain it's not just an engineering limit but a fundamental property of spacetime."},

    {"q": "If you're on a train and jump, why don't you land at the back of the train?",
     "a": "You share the train's velocity when you jump (inertia). You and train move forward together. Must explain reference frames and why your horizontal velocity is preserved."}
]

## 4. Two Prompt Versions

In [6]:
prompts = {

"Version 1": "You are a physics teacher. Answer the question.",

"Version 2": """You are a physics teacher for high school students.

When answering questions:
1. State the physics law/principle first
2. Explain BOTH the ideal case (vacuum/no friction) AND real-world case if they differ
3. Explicitly correct any common misconception related to the question
4. Use simple everyday language with examples

Always address the "trick" or nuance in the question - students often ask questions because they have a misconception."""
}

In [7]:
q1 = dataset[0]["q"]

print(f"\nQUESTION 1:\n{q1}\n")
print("=" * 50)

for prompt_name in ["Version 1", "Version 2"]:
    prompt = prompts[prompt_name]
    response = physics_teacher(prompt, q1)

    print(f"\n{prompt_name} ANSWER:")
    print("-" * 30)
    print(response)


QUESTION 1:
If I drop a bowling ball and a feather from the same height, which hits the ground first and why?


Version 1 ANSWER:
------------------------------
This is a classic physics experiment that demonstrates the concept of air resistance and gravity. 

When you drop a bowling ball and a feather from the same height, they will both accelerate towards the ground due to gravity. However, the key difference lies in the air resistance they experience.

The bowling ball is denser and has a larger mass, which means it has a smaller surface area relative to its mass. As a result, it experiences less air resistance compared to the feather. The feather, on the other hand, is much lighter and has a larger surface area relative to its mass, which means it experiences more air resistance.

Since air resistance acts in the opposite direction of the motion (upward in this case), it slows down the feather more than the bowling ball. As a result, the bowling ball will hit the ground first, whi

## 5. Run Evaluation

In [8]:
results = {name: {"Accuracy": [], "Clarity": [], "Completeness": []}
           for name in prompts}

for prompt_name, prompt in prompts.items():
    print(f"\nEvaluating: {prompt_name}")

    for i, item in enumerate(dataset):
        response = physics_teacher(prompt, item["q"])

        # Get parsed scores from evaluator
        scores = evaluate_response(item["q"], item["a"], response)

        # Store scores safely
        for metric in ["Accuracy", "Clarity", "Completeness"]:
            if metric in scores:
                results[prompt_name][metric].append(scores[metric])
            else:
                results[prompt_name][metric].append(0)  # default if missing

        print(f"  Q{i+1}: {scores}")



Evaluating: Version 1
  Q1: {'Accuracy': 1, 'Clarity': 1, 'Completeness': 1}
  Q2: {'Accuracy': 1, 'Clarity': 1, 'Completeness': 1}
  Q3: {'Accuracy': 0, 'Clarity': 0, 'Completeness': 0}
  Q4: {'Accuracy': 1, 'Clarity': 1, 'Completeness': 1}
  Q5: {'Accuracy': 1, 'Clarity': 1, 'Completeness': 1}
  Q6: {'Accuracy': 0, 'Clarity': 0, 'Completeness': 0}
  Q7: {'Accuracy': 1, 'Clarity': 1, 'Completeness': 0}
  Q8: {'Accuracy': 1, 'Clarity': 1, 'Completeness': 1}
  Q9: {'Accuracy': 1, 'Clarity': 1, 'Completeness': 1}

Evaluating: Version 2
  Q1: {'Accuracy': 1, 'Clarity': 1, 'Completeness': 1}
  Q2: {'Accuracy': 1, 'Clarity': 1, 'Completeness': 1}
  Q3: {'Accuracy': 1, 'Clarity': 1, 'Completeness': 1}
  Q4: {'Accuracy': 1, 'Clarity': 1, 'Completeness': 1}
  Q5: {'Accuracy': 1, 'Clarity': 1, 'Completeness': 1}
  Q6: {'Accuracy': 0, 'Clarity': 0, 'Completeness': 0}
  Q7: {'Accuracy': 1, 'Clarity': 1, 'Completeness': 1}
  Q8: {'Accuracy': 1, 'Clarity': 1, 'Completeness': 1}
  Q9: {'Accuracy': 

## 6. Results Summary (Pass Rates)

In [9]:
print("\n" + "="*50)
print("PASS RATES BY PROMPT VERSION")
print("="*50)

for prompt_name in prompts:
    print(f"\n{prompt_name.upper()}:")
    total_pass = 0
    total_tests = 0
    for metric in ["Accuracy", "Clarity", "Completeness"]:
        passed = sum(results[prompt_name][metric])
        total = len(results[prompt_name][metric])
        total_pass += passed
        total_tests += total
        pct = (100*passed/total) if total > 0 else 0
        print(f"  {metric}: {passed}/{total} ({pct:.0f}%)")
    overall_pct = (100*total_pass/total_tests) if total_tests > 0 else 0
    print(f"  Overall: {total_pass}/{total_tests} ({overall_pct:.0f}%)")


PASS RATES BY PROMPT VERSION

VERSION 1:
  Accuracy: 7/9 (78%)
  Clarity: 7/9 (78%)
  Completeness: 6/9 (67%)
  Overall: 20/27 (74%)

VERSION 2:
  Accuracy: 7/9 (78%)
  Clarity: 7/9 (78%)
  Completeness: 7/9 (78%)
  Overall: 21/27 (78%)
